https://www.youtube.com/watch?v=rsDlu-9UP00&t=7s

https://python.langchain.com/v0.2/docs/integrations/chat/llamacpp/

In [1]:
# %pip install -qU langchain-community llama-cpp-python

In [2]:
local_model = r"C:\Users\vibud\Desktop\Ollama-custom-models\llama-3-1-8b-ins-16bit\Meta-Llama-3.1-8B-Instruct-bf16.gguf"


In [3]:
from langsmith import Client
from dotenv import load_dotenv
import os

load_dotenv()

tavily_api_key = os.getenv("TAVILY_API_KEY")
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# client = Client()
# project_name = "03-01-Ollama-function-calls"
# client.create_project(project_name)
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

In [4]:
import multiprocessing

from langchain_community.chat_models import ChatLlamaCpp

llm = ChatLlamaCpp(
    temperature=0.5,
    model_path=local_model,
    n_ctx=10000,
    n_gpu_layers=8,
    n_batch=300,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    max_tokens=4096,
    n_threads=multiprocessing.cpu_count() - 1,
    repeat_penalty=1.5,
    top_p=0.5,
    verbose=True,
)

llama_model_loader: loaded meta data with 29 key-value pairs and 292 tensors from C:\Users\vibud\Desktop\Ollama-custom-models\llama-3-1-8b-ins-16bit\Meta-Llama-3.1-8B-Instruct-bf16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.licens

In [5]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]

ai_msg = llm.invoke(messages)
ai_msg


llama_print_timings:        load time =    3295.38 ms
llama_print_timings:      sample time =      78.82 ms /   150 runs   (    0.53 ms per token,  1902.97 tokens per second)
llama_print_timings: prompt eval time =    3295.31 ms /    35 tokens (   94.15 ms per token,    10.62 tokens per second)
llama_print_timings:        eval time =   40601.92 ms /   149 runs   (  272.50 ms per token,     3.67 tokens per second)
llama_print_timings:       total time =   44180.32 ms /   184 tokens


AIMessage(content='The translation of "Je m\'appelle Marie" is actually incorrect, it should be:\n\n"I adore le programmation."\n\nHowever if you want an idiomatic expression in french for saying I LOVE something here\'s a more natural way to say that.\n\nIn French we would use the verb AIMER with different intensities. \n\nHere are some examples of how one can express love or strong affection towards programming (or any other subject) :\n\n*   Je m\'appelle Marie adore le programmation.\n    *This is an informal expression, and it\'s not very common to say that you "adore" something in French.)\n    \n<!---->\n\n*Je suis passionnée par la Programmation.*\n \n This one means I am passionate about programming.', response_metadata={'finish_reason': 'stop'}, id='run-0598697f-54d5-40e1-b8a3-22de37f49466-0')

In [6]:
print(ai_msg.content)

The translation of "Je m'appelle Marie" is actually incorrect, it should be:

"I adore le programmation."

However if you want an idiomatic expression in french for saying I LOVE something here's a more natural way to say that.

In French we would use the verb AIMER with different intensities. 

Here are some examples of how one can express love or strong affection towards programming (or any other subject) :

*   Je m'appelle Marie adore le programmation.
    *This is an informal expression, and it's not very common to say that you "adore" something in French.)
    
<!---->

*Je suis passionnée par la Programmation.*
 
 This one means I am passionate about programming.


In [16]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
ai_msg = chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval

llama_print_timings:        load time =    3295.38 ms
llama_print_timings:      sample time =      13.90 ms /    24 runs   (    0.58 ms per token,  1726.49 tokens per second)
llama_print_timings: prompt eval time =    1564.33 ms /    28 tokens (   55.87 ms per token,    17.90 tokens per second)
llama_print_timings:        eval time =    6371.49 ms /    23 runs   (  277.02 ms per token,     3.61 tokens per second)
llama_print_timings:       total time =    7990.92 ms /    51 tokens


In [17]:
print(ai_msg.content)

"Ich liebe Programmieren."

Would you like me translate anything else? Maybe something about your favorite language or framework!


### Tool Calling

In [18]:
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.tools import tool


class WeatherInput(BaseModel):
    location: str = Field(description="The city and state, e.g. San Francisco, CA")
    unit: str = Field(enum=["celsius", "fahrenheit"])


@tool("get_current_weather", args_schema=WeatherInput)
def get_weather(location: str, unit: str):
    """Get the current weather in a given location"""
    return f"Now the weather in {location} is 22 {unit}"


llm_with_tools = llm.bind_tools(
    tools=[get_weather],
    tool_choice={"type": "function", "function": {"name": "get_current_weather"}},
)

In [19]:
ai_msg = llm_with_tools.invoke(
    "what is the weather like in HCMC in celsius",
)
ai_msg

char ::= [^"\] | [\] char_1 
char_1 ::= ["\/bfnrt] | [u] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] 
location-kv ::= ["] [l] [o] [c] [a] [t] [i] [o] [n] ["] space [:] space string 
space ::= space_7 
string ::= ["] string_8 ["] space 
root ::= [{] space location-kv [,] space unit-kv [}] space 
unit-kv ::= ["] [u] [n] [i] [t] ["] space [:] space unit 
space_7 ::= [ ] | 
string_8 ::= char string_8 | 
unit ::= ["] [c] [e] [l] [s] [i] [u] [s] ["] | ["] [f] [a] [h] [r] [e] [n] [h] [e] [i] [t] ["] 


Llama.generate: 2 prefix-match hit, remaining 20 prompt tokens to eval

llama_print_timings:        load time =    3295.38 ms
llama_print_timings:      sample time =     294.58 ms /    19 runs   (   15.50 ms per token,    64.50 tokens per second)
llama_print_timings: prompt eval time =    1126.35 ms /    20 tokens (   56.32 ms per token,    17.76 tokens per second)
llama_print_timings:        eval time =    4895.15 ms /    18 runs   (  271.95 ms per token,     3.68 tokens per second)
llama_print_timings:       total time =    6400.93 ms /    38 tokens


AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{ "location": "Ho Chi Minh City", "unit": "celsius"}'}, 'tool_calls': [{'id': 'call__0_get_current_weather_cmpl-23b7d81c-95c7-4cdf-a623-83516e987b95', 'type': 'function', 'function': {'name': 'get_current_weather', 'arguments': '{ "location": "Ho Chi Minh City", "unit": "celsius"}'}}]}, response_metadata={'token_usage': {'prompt_tokens': 22, 'completion_tokens': 18, 'total_tokens': 40}, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-20a6e60f-ec94-42fd-9e6d-3e7c04188857-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'Ho Chi Minh City', 'unit': 'celsius'}, 'id': 'call__0_get_current_weather_cmpl-23b7d81c-95c7-4cdf-a623-83516e987b95', 'type': 'tool_call'}])

In [22]:
print(ai_msg.content)
# print(ai_msg. additional_kwargs.function_call.arguments)

print("Tool Name:", ai_msg.additional_kwargs['function_call']['name'])
print("Arguments:", ai_msg.additional_kwargs['function_call']['arguments'])



Tool Name: get_current_weather
Arguments: { "location": "Ho Chi Minh City", "unit": "celsius"}


In [14]:
ai_msg = llm_with_tools.invoke(
    "what is the weather like in Vanarasi in celsius",)

print(ai_msg.content)

char ::= [^"\] | [\] char_1 
char_1 ::= ["\/bfnrt] | [u] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] 
location-kv ::= ["] [l] [o] [c] [a] [t] [i] [o] [n] ["] space [:] space string 
space ::= space_7 
string ::= ["] string_8 ["] space 
root ::= [{] space location-kv [,] space unit-kv [}] space 
unit-kv ::= ["] [u] [n] [i] [t] ["] space [:] space unit 
space_7 ::= [ ] | 
string_8 ::= char string_8 | 
unit ::= ["] [c] [e] [l] [s] [i] [u] [s] ["] | ["] [f] [a] [h] [r] [e] [n] [h] [e] [i] [t] ["] 


Llama.generate: 21 prefix-match hit, remaining 1 prompt tokens to eval

llama_print_timings:        load time =    3295.38 ms
llama_print_timings:      sample time =     273.94 ms /    18 runs   (   15.22 ms per token,    65.71 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (-nan(ind) ms per token, -nan(ind) tokens per second)
llama_print_timings:        eval time =    4964.07 ms /    18 runs   (  275.78 ms per token,     3.63 tokens per second)
llama_print_timings:       total time =    5325.90 ms /    18 tokens


In [15]:
print(ai_msg.content)